In [31]:
import pandas as pd
import numpy as np

In [32]:
# =====================================================
# Load Dataset & Basic Preprocessing
# =====================================================

df = pd.read_csv('../Data/uber_taxi_demand_5years.csv')
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.sort_values('datetime').reset_index(drop=True)

df['time_index'] = (df['datetime'] - df['datetime'].min()).dt.total_seconds() / 3600
df.set_index('datetime', inplace=True)   

df.head()

,zone,pickup_count,temperature,precipitation,is_holiday,is_raining,time_index
datetime,,,,,,,
2021-01-01,Airport,42,1.7,0.0,1,0,0.0
2021-01-01,Downtown,77,1.7,0.0,1,0,0.0
2021-01-01,Midtown,16,1.7,0.0,1,0,0.0
2021-01-01,Suburb,3,1.7,0.0,1,0,0.0
2021-01-01,Uptown,12,1.7,0.0,1,0,0.0


In [33]:
df['hours'] = df.index.hour
df['hour_sin'] = np.sin(2 * np.pi * df['hours'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hours'] / 24)

df['day_of_month'] = df.index.day

df["day_of_week"] = df.index.dayofweek

df["month"] = df.index.month
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

df["quarter"] = df.index.quarter

df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

In [34]:
df['lag_1'] = df.groupby('zone')['pickup_count'].shift(1)
df['lag_2'] = df.groupby('zone')['pickup_count'].shift(2)
df['lag_3'] = df.groupby('zone')['pickup_count'].shift(3)
df['lag_24'] = df.groupby('zone')['pickup_count'].shift(24)
df['lag_168'] = df.groupby('zone')['pickup_count'].shift(168)

In [35]:
df['rolling_mean_24']  = df.groupby('zone')['pickup_count'].transform(lambda x: x.shift(1).rolling(24).mean())
df['rolling_mean_168'] = df.groupby('zone')['pickup_count'].transform(lambda x: x.shift(1).rolling(168).mean())

In [36]:
df['expanding_sum'] = (
    df.groupby('zone')['pickup_count']
      .transform(
          lambda x: x.shift(1)
                    .expanding()
                    .sum()
      )
)

df['expanding_mean'] = (
    df.groupby('zone')['pickup_count']
      .transform(
          lambda x: x.shift(1)
                    .expanding()
                    .mean()
      )
)

df['expanding_std'] = (
    df.groupby('zone')['pickup_count']
      .transform(
          lambda x: x.shift(1)
                    .expanding()
                    .std()
      )
)

In [37]:
df["zone_data"] = df["zone"]
df = pd.get_dummies(df, columns=['zone'],drop_first=True, dtype=int)

In [38]:
print(f"Total features created: {len(df.columns)}") 
print("\nFeature columns:")
print(df.columns.tolist()) 

Total features created: 31

Feature columns:
['pickup_count', 'temperature', 'precipitation', 'is_holiday', 'is_raining', 'time_index', 'hours', 'hour_sin', 'hour_cos', 'day_of_month', 'day_of_week', 'month', 'month_sin', 'month_cos', 'quarter', 'is_weekend', 'lag_1', 'lag_2', 'lag_3', 'lag_24', 'lag_168', 'rolling_mean_24', 'rolling_mean_168', 'expanding_sum', 'expanding_mean', 'expanding_std', 'zone_data', 'zone_Downtown', 'zone_Midtown', 'zone_Suburb', 'zone_Uptown']


In [39]:
df_clean = df.dropna()
print(f"\nOriginal rows : {len(df)}")
print(f"After dropna  : {len(df_clean)}")
print(f"Rows removed  : {len(df) - len(df_clean)}")
print(f"Data loss %   : {((len(df) - len(df_clean)) / len(df)) * 100:.1f}%")
df = df.dropna()


Original rows : 219120
After dropna  : 218280
Rows removed  : 840
Data loss %   : 0.4%


In [40]:
df.head(5)

,pickup_count,temperature,precipitation,is_holiday,is_raining,time_index,hours,hour_sin,hour_cos,day_of_month,...,rolling_mean_24,rolling_mean_168,expanding_sum,expanding_mean,expanding_std,zone_data,zone_Downtown,zone_Midtown,zone_Suburb,zone_Uptown
datetime,,,,,,,,,,,,,,,,,,,,,
2021-01-08,13,-4.2,0.0,0,0,168.0,0,0.0,1.0,8,...,21.000000,18.446429,3099.0,18.446429,14.539543,Uptown,0,0,0,1
2021-01-08,6,-4.2,0.0,0,0,168.0,0,0.0,1.0,8,...,8.250000,6.196429,1041.0,6.196429,5.771351,Suburb,0,0,1,0
2021-01-08,92,-4.2,0.0,0,0,168.0,0,0.0,1.0,8,...,40.041667,39.113095,6571.0,39.113095,28.905468,Downtown,1,0,0,0
2021-01-08,72,-4.2,0.0,0,0,168.0,0,0.0,1.0,8,...,50.041667,44.809524,7528.0,44.809524,15.592088,Airport,0,0,0,0
2021-01-08,29,-4.2,0.0,0,0,168.0,0,0.0,1.0,8,...,56.500000,49.339286,8289.0,49.339286,39.656584,Midtown,0,1,0,0


In [41]:
corr = df.corr(numeric_only=True)
corr['pickup_count'].sort_values(ascending=False)

pickup_count        1.000000
lag_168             0.914659
lag_1               0.843096
lag_24              0.802631
lag_2               0.650482
rolling_mean_24     0.613312
rolling_mean_168    0.587690
expanding_mean      0.577267
expanding_std       0.487310
expanding_sum       0.446677
lag_3               0.420115
zone_Midtown        0.309096
hours               0.181053
is_raining          0.154242
zone_Downtown       0.136546
precipitation       0.129112
time_index          0.118484
temperature         0.099232
quarter             0.046027
month               0.044900
is_holiday          0.017770
day_of_month        0.004498
month_cos          -0.039966
month_sin          -0.070513
hour_sin           -0.105979
hour_cos           -0.129337
day_of_week        -0.156783
is_weekend         -0.220459
zone_Uptown        -0.220558
zone_Suburb        -0.436849
Name: pickup_count, dtype: float64

In [42]:
features = [
    'zone_data',
    'pickup_count',
    'time_index',

    # Lag features
    'lag_1',
    'lag_2',
    'lag_3',
    'lag_24',
    'lag_168',

    # Rolling features
    'rolling_mean_24',
    'rolling_mean_168',

    # Time features
    'hour_sin',
    'hour_cos',
    'day_of_week',
    'is_weekend',
    'month_sin',
    'month_cos',

    # Weather / external features
    'temperature',
    'precipitation',
    'is_raining',
    'is_holiday',

    # Zone features
    'zone_Downtown',
    'zone_Midtown',
    'zone_Suburb',
    'zone_Uptown'
]

In [44]:
cols_to_save = features
df_save = df[cols_to_save].copy()

# Save
df_save.to_csv('feature-columns.csv', index=True)  

print(f"✅ Saved!")
print(f"Shape: {df_save.shape}")

✅ Saved!
Shape: (218280, 24)
